In [3]:
using ITensors, ITensorMPS, LinearAlgebra, HDF5
ITensors.disable_threaded_blocksparse() #Disables block sparse multithreading
BLAS.set_num_threads(4) #Enables multithreading with BLAS

### Initial Parameters ###
N = 61 #Number of lattice sites per dimension
d = 1 #Number of spatial dimensions
Dim = 10 #Truncated local Hilbert space dimension
a = 0.5#1.0 #Lattice spacing

n_0 = round(Int, ((N-1)/2)) #Index of the point at the center of the lattice (0-based indexing).

mass = 1.0
m0 = 1.0 #Basis frequency
l = 0.1 #phi^4 coupling strength

### Field Operator $\phi(\mathbf{x})$ and $\pi(\mathbf{x})$ ###
function a_matrix(D)
    A = zeros(D, D)
    for n in 1:D-1
        A[n, n+1] = sqrt(n)
    end
    return A
end

phi_matrix = D -> (a_matrix(D) + a_matrix(D)')/sqrt(2*m0)
pi_matrix = D -> im*sqrt(m0/2)*(a_matrix(D)' - a_matrix(D))

ITensors.op(::OpName"phi", ::SiteType"Boson", D::Int) = phi_matrix(D)

ITensors.op(::OpName"phi2", ::SiteType"Boson", D::Int) = phi_matrix(D)^2

ITensors.op(::OpName"phi4", ::SiteType"Boson", D::Int) = phi_matrix(D)^4

ITensors.op(::OpName"pi", ::SiteType"Boson", D::Int) = pi_matrix(D)

ITensors.op(::OpName"pi2", ::SiteType"Boson", D::Int) = pi_matrix(D)^2

### Hamiltonian ###
H0 = OpSum() #Non-interacting Hamiltonian OpSum
for x in 1:N
    global H0 += a^d/2, "pi2", x
    global H0 += a^d * d/a^2, "phi2", x
    if x < N
        global H0 -= a^d * 1/a^2, "phi", x, "phi", x+1
    else
        global H0 -= a^d * 1/a^2, "phi", x, "phi", 1
    end
    global H0 += a^d/2 * mass^2, "phi2", x
end

HInt = OpSum() #Interacting Hamiltonian OpSum
for x in 1:N
    global HInt += l/factorial(4) * a^d, "phi4", x
end

H_OS = H0 + HInt #Full Hamiltonian OpSum

sites = siteinds("Boson", N; dim=Dim); #Create ITensor sites

H = MPO(H_OS, sites); #Hamiltonian MPO

### Vacuum State MPS ###
psi0 = random_mps(sites;linkdims=10)
nsweeps = 50
maxdim = [50, 50, 100, 100, 200, 200]
cutoff = [1E-10]
energy, vac = dmrg(H,psi0;nsweeps,maxdim,cutoff)

EEC_MPS = h5open("Z:/Energy Correlator/1d_EEC_MPS/N=$N,a=$a,dim=$Dim,l=$l,m=$mass", "w")
write(EEC_MPS, "sites", sites)
write(EEC_MPS, "vac", vac)
close(EEC_MPS)

After sweep 1 energy=198.68511849709742  maxlinkdim=50 maxerr=2.78E-05 time=29.757
After sweep 2 energy=150.0411626365737  maxlinkdim=50 maxerr=5.28E-06 time=41.859
After sweep 3 energy=118.1013412830315  maxlinkdim=100 maxerr=7.18E-07 time=156.557
After sweep 4 energy=88.7770270100209  maxlinkdim=100 maxerr=1.59E-07 time=179.336
After sweep 5 energy=71.50712150737695  maxlinkdim=200 maxerr=3.32E-10 time=528.348
After sweep 6 energy=62.7288024492864  maxlinkdim=200 maxerr=5.90E-09 time=781.777
After sweep 7 energy=57.75060939168489  maxlinkdim=200 maxerr=3.91E-10 time=807.131
After sweep 8 energy=55.075581040183934  maxlinkdim=200 maxerr=1.65E-10 time=739.003
After sweep 9 energy=53.06375088879355  maxlinkdim=199 maxerr=9.99E-11 time=623.014
After sweep 10 energy=51.33100682411959  maxlinkdim=195 maxerr=9.99E-11 time=488.574
After sweep 11 energy=49.92727208920705  maxlinkdim=191 maxerr=9.98E-11 time=409.897
After sweep 12 energy=48.753016973917376  maxlinkdim=184 maxerr=9.97E-11 time=

In [4]:
@show linkdims(vac);

linkdims(vac) = [8, 11, 13, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 14, 15, 15, 14, 14, 17, 16, 15, 14, 16, 14, 15, 14, 14, 17, 16, 18, 16, 16, 15, 15, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 11, 8]
